# EX_02 — Embeddings con Transformers (ejercicios)

**Notebook de referencia:** `notebook/02_Embeddings_Transformers.ipynb`

**Tiempo orientativo:** ~30 minutos.


## Actividad 1 — Mean pooling

Con `AutoTokenizer` + `AutoModel`, obtén **last_hidden_state** para una frase y calcula el embedding de frase como media sobre tokens (excluyendo padding).


In [1]:
from transformers import AutoTokenizer, AutoModel
import torch

text = "Transformers build contextual embeddings."

# Modelo pequeño y ligero
model_name = "distilbert-base-uncased"

# Tokenizer y modelo
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

# Tokenización
inputs = tokenizer(
    text,
    return_tensors="pt",
    padding=True,
    truncation=True
)

# Forward pass
with torch.no_grad():
    outputs = model(**inputs)

# Hidden states
token_embeddings = outputs.last_hidden_state

# Attention mask para ignorar padding
attention_mask = inputs["attention_mask"]

# Mean pooling ignorando padding
mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()

sum_embeddings = torch.sum(token_embeddings * mask_expanded, dim=1)
sum_mask = torch.clamp(mask_expanded.sum(dim=1), min=1e-9)

sentence_embedding = sum_embeddings / sum_mask

print("Sentence embedding shape:")
print(sentence_embedding.shape)

print("\nSentence embedding:")
print(sentence_embedding)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

C:\Users\jassa\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\jassa\.cache\huggingface\hub\models--distilbert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Sentence embedding shape:
torch.Size([1, 768])

Sentence embedding:
tensor([[ 1.5190e-01,  2.9744e-02, -2.3500e-02,  1.2730e-01, -3.1240e-02,
         -8.0350e-02, -2.1901e-01,  2.5923e-01,  2.2741e-01, -3.8661e-01,
          2.4434e-02,  3.4409e-03, -3.4283e-01,  2.7498e-01, -3.8251e-01,
          4.7404e-02, -2.2486e-01,  6.9558e-02, -9.9181e-02,  6.9389e-02,
         -6.0562e-02,  2.8674e-02, -5.1416e-02,  4.8923e-01,  2.0923e-01,
          3.4799e-02,  2.1264e-01,  2.4067e-01, -2.2958e-01,  1.2813e-01,
          2.6159e-02,  3.6401e-01, -2.1773e-01, -4.0076e-01,  3.9241e-02,
          6.9977e-02,  3.4154e-02, -1.7549e-01, -1.0747e-01,  6.1282e-02,
         -5.0228e-01, -2.5912e-01,  8.5448e-02,  1.7959e-02, -2.2127e-01,
         -2.1835e-01, -3.4958e-01, -1.1215e-01, -1.6745e-01, -1.9101e-01,
         -7.9652e-01,  1.1566e-01, -3.9611e-02,  2.5552e-01,  1.4628e-01,
          6.7055e-01,  1.3695e-01, -5.6156e-01,  3.9177e-01, -2.3554e-02,
         -1.0667e-01,  1.4320e-01, -6.2761e-

## Actividad 2 — `sentence-transformers`

Usa `SentenceTransformer` para embedder dos frases y calcula similitud coseno. Comenta brevemente (en inglés en un comentario) por qué suele ser mejor que mean-pooling manual de BERT base.


In [2]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Load lightweight sentence-transformers model
model = SentenceTransformer("all-MiniLM-L6-v2")

# Example sentences
sentence1 = "Artificial intelligence is transforming many industries."
sentence2 = "AI is changing the way companies operate."

# Generate embeddings
embedding1 = model.encode(sentence1)
embedding2 = model.encode(sentence2)

# Cosine similarity
cosine_similarity = np.dot(embedding1, embedding2) / (
    np.linalg.norm(embedding1) * np.linalg.norm(embedding2)
)

print("Cosine similarity:")
print(cosine_similarity)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Cosine similarity:
0.7843232


## Actividad 3 — Paráfrasis

Escribe dos paráfrasis de una misma idea y muestra que sus embeddings (sentence-transformers) tienen **mayor** similitud entre sí que con una frase de tema distinto.


In [3]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Load model
model = SentenceTransformer("all-MiniLM-L6-v2")

# Sentences
sentence1 = "The cat is sleeping on the sofa."
sentence2 = "A cat is resting on the couch."
sentence3 = "Stock markets were highly volatile today."

# Generate embeddings
emb1 = model.encode(sentence1)
emb2 = model.encode(sentence2)
emb3 = model.encode(sentence3)

# Cosine similarity function
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# Similarities
sim_paraphrase = cosine_similarity(emb1, emb2)
sim_unrelated = cosine_similarity(emb1, emb3)

print("Similarity between paraphrases:")
print(sim_paraphrase)

print("\nSimilarity with unrelated sentence:")
print(sim_unrelated)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Similarity between paraphrases:
0.7294781

Similarity with unrelated sentence:
-0.08222487
